Mirar en Sitra puntos de regulación comerciales

IP=ftp://10.0.6.78/PRO/  
User=GENMES1  
Pass=DeimosTT  

Buenas, se me ha ocurrido una idea de informe que creo nos podría ayudar a tener una visión de conjunto de la calidad de la info que publica Circulación que tenga que ver con el SIV. Similar a los informes de puntualidad para, en el futuro, poder comparar si vamos mejorando.
Por ejemplo, el informe podría ser por núcleo de Cercanías, línea o estación.
Indicadores a mostrar:
- Trenes especiales por día, semana, mes
- Trenes suprimidos en origen
- Trenes suprimidos en trayecto
- Anticipación de vías en origen (anticipación de la rotulación)
- Tiempos medios de Inferencia de vía de CTC (por vía, por relación según acceso a estación...)
- Cumplimiento de vía planificada (por vía, por destino...?)
- Calidad de vía planificada mostrada en SIV
- Calidad de mensajería xSIV por estación/día
    - en origen: alta y salida
    - de paso: aprox, entrada, salida
    - en destino: aprox, entrada

- Trenes especiales por día, semana, mes:  
    Cuál es el criterio exactamente para `tren especial` (material vacío, servicio interno?)
- Trenes suprimidos  
    Identificación tren suprimido (mensaje xSIV?)
    - En origen
    - En trayecto
- Calidad de mensajería xSIV por estación/día
    - en origen: alta y salida
    - de paso: aprox, entrada, salida
    - en destino: aprox, entrada
- Anticipación de vías en origen (anticipación de la rotulación)  
    Se usa la `calidad de mensajería` del apartado anterior
- Tiempos medios de Inferencia de vía de CTC (por vía, por relación según acceso a estación...)  
    `Llegada` - `Anticipación` ?
- Cumplimiento de vía planificada (por vía, por destino...?)  
    Importar estadísticas fiabilidad
- Calidad de vía planificada mostrada en SIV  
    Importar estadísticas fiabilidad

# Import

In [ ]:
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
import json
from collections import Counter
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import regex
import yaml
from lxml import etree
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor
from src.processor.log_procesor import LogProcessor
from src.utils import formatTimedelta  # loadViasFromTopos,
from src.utils import (
    dateFromText,
    getEstacionamientos,
    getFilesByDate,
    getFilesByWeek,
    getNumbers,
    guardarExcel,
    guardarExcelMulti,
    isEmpty,
    isValidCode,
    listTopos,
    loadEstaciones,
    localizeFecha,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    rellenarId,
    roundGroup,
    setEF,
    slidingWindow,
    sortElements,
    sortStrNumbers,
    splitDataframe,
    splitList,
    splitLongString,
    time2localtime,
)
from src.visualizacion.visualizaciones import (
    build_hierarchical_dataframe,
    sample_random_colors,
    setHoverInfo,
)

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}

In [ ]:
start_date = "2025-07-04"
end_date = "2025-07-05"

# Carga Fuentes

In [ ]:
traficos = {
    "OTROS": [
        "MAQUINA AISLADA",
        "Mercancias",
        "Material vacio RAM",
        "Mercancias RAM",
        "Servicio Interno",
        "Material Vacio",
        "Maquina Aislada",
        "Maquina Aislada Mercancias",
        "Transporte excepcional",
        "T.L.E.",
    ],
    "ESPECIALES": [" "],
    "CERCANÍAS": [
        "CERCANIAS",
        "CERCANIAS RAM",
    ],
    "MD/LD/AV": [
        "REGIONAL EXPRES",
        "REGIONAL RAM",
        "MD",
        "INTERCITY",
        "INTERURBANO",
        "AVE",
        "ALVIA",
        "AVLO",
        "AVANT",
        "IRYO",
        "LANZADERA-MIXTA",
        "OUIGO",
        "TALGO",
        "EUROMED",
        "VIAJEROS",
        "Viajeros Larga Distancia y AV",
        "Media Distancia CP",
    ],
}

## ControlPoints

In [ ]:
control_points = []

for fname in Path(r"data/tablas auxiliares/").glob(r"controlPointTable*.xml"):
    tree = etree.parse(fname)
    root = tree.getroot()
    for cpoint in root.iterchildren():
        # info = dict()
        info = []
        info.extend(cpoint.items())
        for el in cpoint.iterchildren():
            # info[el.tag.split("}")[-1]] = el.values()[0]
            info.append((el.tag.split("}")[-1], el.values()[0]))
        control_points.append(dict(info))
control_points = (
    pd.DataFrame(control_points)
    .drop_duplicates(subset=["code", "shortDesc", "KMPoint"])
    .reset_index(drop=True)
)
# estaciones = sorted(loadEstaciones()["Código"].unique().tolist())
estaciones = sorted(control_points["code"].unique().tolist())

## Histórico MOW

In [ ]:
ntrenes = [rellenarId(el) for el in np.arange(100000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    [],
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

## XSIV

In [ ]:
# source = "xsiv"
# dir_logs = Path(f"C:/Users/jose.espinosa/Documents/Data/{source}/PRO")


# estaciones = []

# log_processor = LogProcessor()
# w_logs = getFilesByDate(dir_logs, start_date, end_date)
# if not w_logs:
#     print("No hay logs en los días seleccionados")
# else:
#     fnames, full_days = list(zip(*w_logs))
#     days = f"{full_days[0].strftime('%Y-%m-%d')} - {full_days[-1].strftime('%Y-%m-%d')}"
#     # Procesamos
#     df_logs = log_processor.loadFilesLogs(
#         fnames,
#         source,
#         train_types=train_types,
#         days=days,
#         estaciones=estaciones,
#         format_fechas=True,
#     )
#     # df_logs = df_logs[
#     #     (df_logs["Fecha"] >= pd.to_datetime(start_date))
#     #     & (df_logs["Fecha"] <= pd.to_datetime(end_date))
#     # ]
# # Tomo que hace la aproximación, pero esta no tiene por qué ser correcta, así que descarto su vía
# df_logs.loc[df_logs["Movimiento"] == "APROXIMACIÓN", "Vía"] = np.nan
# df_logs.loc[df_logs["Vía"].apply(isEmpty), "Vía"] = None

# # Añadir información de la fecha
# df_logs = df_logs[
#     (df_logs["Desconocido"] == "false") & (df_logs["NTécnico"].apply(isValidCode))
# ].reset_index(drop=True)
# df_logs["Día"] = df_logs["Fecha"].dt.date
# df_logs["day_of_week"] = df_logs["Fecha"].dt.day_of_week
# df_logs["day_of_year"] = df_logs["Fecha"].dt.day_of_year
# df_logs["week_of_year"] = (df_logs["day_of_year"] / 7).astype(int)

# df_logs["_mov"] = df_logs["Movimiento"].apply(mov_sorter.get)

## Fuentes de Vía

In [ ]:
fuentes = getFilesByDate(Path("data/FuentesVías"), start_date, end_date, ftype="xlsx")
resumen = []
computo_estacion = []
fuentes_vias = []
detalle_tren = []

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with tqdm(total=len(fuentes)) as pbar:
        for f, _ in fuentes:
            fecha = dateFromText(f.stem)
            pbar.set_description(f"{fecha}")

            excel_info = pd.read_excel(f, sheet_name=None, engine="openpyxl")

            r = excel_info["Resumen"].copy()
            r.columns = r.iloc[0].tolist()

            r = r.loc[1:, r.columns[1:]]
            r["Fecha"] = fecha
            resumen.append(r)

            ce = excel_info["CómputoEstación"].copy()
            ce.columns = ce.iloc[2].tolist()

            ce = ce.loc[3:, ce.columns[1:]]
            ce["Fecha"] = fecha
            computo_estacion.append(ce)

            ev = excel_info["CómputoEstaciónVía"].copy()
            ev.columns = ev.iloc[2].tolist()

            ev = ev.loc[3:, ev.columns[1:]]
            ev["Fecha"] = fecha
            fuentes_vias.append(ev)

            dt = excel_info["DetalleTren"].copy()
            dt.columns = dt.iloc[0].tolist()

            dt = dt.loc[1:, dt.columns[1:]]
            detalle_tren.append(dt)

            pbar.update()

resumen = pd.concat(resumen).reset_index(drop=True)
computo_estacion = pd.concat(computo_estacion).reset_index(drop=True)
fuentes_vias = pd.concat(fuentes_vias).reset_index(drop=True)
detalle_tren = pd.concat(detalle_tren).reset_index(drop=True)
detalle_tren[["Cod. Est", "Tren"]] = detalle_tren[["Cod. Est", "Tren"]].map(rellenarId)

In [ ]:
subdirecciones = (
    computo_estacion[
        [
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by=[
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
        ]
    )
    .reset_index(drop=True)
)

## Cargar zona

In [ ]:
# estaciones = ["17000", "18000"]
# zona = "Asturias"
# estaciones = []
# with Path(f"data/Orden movimientos/{zona}.yaml").open("r", encoding="utf8") as f:
#     lineas = yaml.safe_load(f)
# for e in lineas.values():
#     estaciones.extend(e)
# estaciones = sorted(list(set(estaciones)))
# estaciones = control_points["code"][control_points["code"].apply(isValidCode)].tolist()

In [ ]:
# detalle_movimientos_todo: pd.DataFrame = (
#     df_logs[
#         (df_logs["Desconocido"] == "false")
#         & (df_logs["NTécnico"].apply(isValidCode))
#         & (df_logs["Código"].isin(estaciones))
#         # & (df_logs["Día"] == df_logs["Día"].unique()[0])
#         # & (
#         #     df_logs["Producto"].isin(
#         #         ["CERCANIAS", "AVE", "MD", "ALVIA", "OUIGO", "IRYO"]
#         #         + ["AVANT", "REGIONAL EXPRES", "INTERCITY", "INTERURBANO"]
#         #     )
#         # )
#     ][
#         [
#             "Fecha",
#             "Movimiento",
#             "Producto",
#             "FechaOrigen",
#             "NTécnico",
#             "Secuencia",
#             "Nombre",
#             "Código",
#             "Vía",
#             "Estado",
#             "Día",
#             "_mov",
#             # "FuenteMovimiento",
#             # "VíaPlanificada",
#         ]
#     ]
#     .sort_values(by=["FechaOrigen", "NTécnico", "Fecha", "_mov"])
#     .reset_index(drop=True)
#     .drop(["_mov"], axis=1)
# )

In [ ]:
# subdireccion = "Centro"
# trafico = "CERCANÍAS"
# sub_codes = subdirecciones.loc[subdirecciones["Subdirección"] == subdireccion, "Código"]

# detalle_movimientos = pd.merge(
#     detalle_movimientos_todo[
#         detalle_movimientos_todo["Producto"].isin(traficos[trafico])
#     ],
#     subdirecciones[["Código", "Provincia", "Subdirección"]],
#     how="inner",
#     on="Código",
# )

## Cálculos

In [ ]:
detalle_movimientos = historico_pro.copy()

### Rotulaciones/Desrotulaciones incorrectas

In [ ]:
# Nos quedamos unicamente con las circulaciones que tengan origen en la misma fecha
# (de momento nos quitamos potenciales errores)
aux_df = historico_pro[
    historico_pro["FechaOrigen"] == historico_pro["Fecha"].dt.date
].copy()

aux_split = np.split(
    aux_df,
    np.where(
        (~aux_df["NTécnico"].eq(aux_df["NTécnico"].shift()))
        | (~aux_df["FechaOrigen"].eq(aux_df["FechaOrigen"].shift()))
    )[0][1:],
)
cols = [
    "FechaOrigen",
    "CTC",
    "NTécnico",
    "LíneaComercial",
    "Secuencia",
    "Código",
    "Nombre",
    "Movimiento",
    "CategoríaCirculación",
    "Producto",
    "Empresa",
]
origenes = []
destinos = []
continuaciones =  []
ultimos = []
for df in aux_split:
    # Comprobamos las que no tienen origen correcto
    if not any(df["Secuencia"] == 1) and not (
        df.shape[0] == 1 and df["Movimiento"].iloc[0] == "PÉRDIDA_SEGUIMIENTO"
    ):
        # Si es una pérdida de seguimiento, la descartamos
        # continue
        # origenes.append(df)
        if not (df["Movimiento"] == "ORIGEN").any():
            origenes.append(df.iloc[0][cols])
    # Comprobamos que la circulación llegue a destino
    if any(
        (df["Movimiento"].isin(["FIN", "BAJA"])) & (df["Código"] == df["CódigoDestino"])
    ):
        # Nos quedamos con el destino
        df = df.reset_index(drop=True)
        destino = df[
            (df["Movimiento"].isin(["FIN", "BAJA"]))
            & (df["Código"] == df["CódigoDestino"])
        ].iloc[-1]
        # Si el último movimiento es la finalización, pasamos
        if df.shape[0] == destino.name + 1:
            continue
        continuacion = df.iloc[destino.name + 1 :]
        if any(
            np.invert(continuacion["Código"] == destino["Código"])
            & (continuacion["Secuencia"] == -1)
        ): 
            destinos.append(destino)
            # destinos.append(continuacion.iloc[0][cols])
            continuaciones.append(continuacion)
            ultimo = continuacion.iloc[continuacion.shape[0]-1]
            if (ultimo.Código != destino["Código"]):
                ultimos.append(ultimo)
                


In [ ]:
origenes

In [ ]:
destinos = pd.DataFrame(destinos)
destinos = destinos[["FechaOrigen", "CTC", "NTécnico", "LíneaComercial","Secuencia", "Código", "Nombre","Movimiento","CategoríaCirculación","Producto","Empresa"]]
destinos = destinos.sort_values(by=["FechaOrigen", "NTécnico"])



In [ ]:
ultimos = pd.DataFrame(ultimos)
ultimos.reset_index(drop=True, inplace=True)
ultimos.sort_values(by=["FechaOrigen", "NTécnico"])


In [ ]:
info_extra = ultimos[["NTécnico", "FechaOrigen", "Nombre", "Código"]]

In [ ]:
info_extra = info_extra.rename(columns={"Nombre": "ESTACIÓN HASTA LA QUE SIGUE ROTULADO", "Código":"CÓDIGO ESTACIÓN DESROTULAN"})

In [ ]:
rename_column = {"Secuencia": "SECUENCIA DONDE FINALIZA","Nombre": "ESTACIÓN EN LA QUE FINALIZA", "Código":"CÓDIGO ESTACIÓN FINALIZA"}
destinos.rename(columns=rename_column, inplace=True)

In [ ]:

df_merge = pd.merge(
    destinos,
    info_extra,
    how="left",
    on = ["NTécnico", "FechaOrigen"]
)

In [ ]:
estaciones=loadEstaciones()
estaciones = estaciones[["CTC","NombreCTC"]]
estaciones.drop_duplicates(inplace=True)
origenes= pd.DataFrame(origenes)
origenes = pd.merge(
    origenes,
    estaciones,
    how="left",
    on=["CTC"]
)
origenes.drop(columns="CTC",inplace=True)


In [ ]:
origenes

In [ ]:
df_merge = pd.merge(
    df_merge,
    estaciones,
    how="left",
    on = ["CTC"]
)
df_merge.drop(columns="CTC", inplace=True)

In [ ]:
df_merge=df_merge[['FechaOrigen','NombreCTC','NTécnico','LíneaComercial','Producto','SECUENCIA DONDE FINALIZA','CÓDIGO ESTACIÓN FINALIZA','ESTACIÓN EN LA QUE FINALIZA','ESTACIÓN HASTA LA QUE SIGUE ROTULADO','CÓDIGO ESTACIÓN DESROTULAN','Movimiento','CategoríaCirculación','Empresa']]

In [ ]:
origenes = pd.DataFrame(origenes)
rename_column_origen = {"Secuencia":"Secuencia en la que se rotula"}
origenes.rename(columns=rename_column_origen,inplace=True)


In [ ]:
origenes


In [ ]:
from datetime import datetime
origenes["FechaOrigen"] = origenes["FechaOrigen"].dt.date
origenes = origenes.sort_values(by=["FechaOrigen", "NTécnico"])
df_merge["FechaOrigen"] = df_merge["FechaOrigen"].dt.date
df_merge = df_merge.sort_values(by=["FechaOrigen", "NTécnico"])
# guardarExcel(
#     origenes,
#     "Rotulaciones erroneas.xlsx",
#     "Origen",
#     False,
# )
# guardarExcel(
#     destinos,
#     "Rotulaciones erroneas.xlsx",
#     "Destino",
#     True,
# )
today_str = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\Rotulacion") / f"{today_str}_rotulaciones_erroneas.xlsx"

# data = {"trenes_no_rotulado_en_el_origen": origenes, "trenes_que_no_se_desrotulan": df_merge}
# guardarExcelMulti(data,fname)



### Trenes especiales

In [ ]:
# Todos los especiales
especiales = detalle_movimientos[
    detalle_movimientos["NTécnico"].apply(
        lambda x: isValidCode(x) and regex.search(r"^9\d{4}$", x) is not None
    )
].fillna(value={"Producto": ""})


# Específicos
trenes_especiales_estacion = (
    especiales.loc[
        (especiales["Código"].isin(estaciones)),
        ["Día", "NTécnico", "Código", "Nombre"],
    ]
    .drop_duplicates()
    .groupby(["Día", "Código", "Nombre"])
    .size()
    .reset_index()
    .rename(columns={0: "TrenesEspeciales"})
    .sort_values(by=["Día", "Código", "Nombre", "TrenesEspeciales"])
    .copy()
)

trenes_especiales_dia = (
    especiales.loc[
        (especiales["Código"].isin(estaciones)),
        ["Día", "NTécnico"],
    ]
    .drop_duplicates()
    .groupby(["Día"])
    .size()
    .reset_index()
    .rename(columns={0: "TrenesEspeciales"})
    .sort_values(by=["Día"])
    .copy()
)

In [ ]:
display(trenes_especiales_estacion.head())
display(trenes_especiales_dia.head())

### Trenes Suprimidos

In [ ]:
# Todos los suprimidos
suprimidos = (
    detalle_movimientos[
        # (detalle_movimientos["Estado"].isin(["SUPPRESSED", "RUNNING_AFTER_SUPPRESSED"]))
        (detalle_movimientos["Movimiento"] == "ELIMINACIÓN")
    ]
    .drop_duplicates(
        subset=[
            "FechaOrigen",
            "NTécnico",
            "Secuencia",
            "Nombre",
            "Código",
            "Movimiento",
        ],
        keep="first",
    )
    .sort_values(by=["FechaOrigen", "NTécnico", "Fecha"])
    .reset_index(drop=True)
    .copy()
)

# Específicos
trenes_suprimidos_estacion = (
    suprimidos.loc[
        (suprimidos["Código"].isin(estaciones)),
        ["Día", "NTécnico", "Código", "Nombre"],
    ]
    .drop_duplicates()
    .groupby(["Día", "Código", "Nombre"])
    .size()
    .reset_index()
    .rename(columns={0: "TrenesSuprimidos"})
    .sort_values(by=["Día", "TrenesSuprimidos"], ascending=[True, False])
    .copy()
)

trenes_suprimidos_dia = (
    suprimidos.loc[
        (suprimidos["Código"].isin(estaciones)),
        ["Día", "NTécnico"],
    ]
    .drop_duplicates()
    .groupby(["Día"])
    .size()
    .reset_index()
    .rename(columns={0: "TrenesSuprimidos"})
    .sort_values(by=["Día"], ascending=[True])
    .copy()
)

In [ ]:
display(trenes_suprimidos_estacion.head())
display(trenes_suprimidos_dia.head())

#### En Origen

In [ ]:
# Específicos
trenes_suprimidos_estacion_origen = (
    suprimidos.loc[
        (suprimidos["Código"].isin(estaciones)) & (suprimidos["Secuencia"] == 1),
        ["Día", "NTécnico", "Código", "Nombre"],
    ]
    .drop_duplicates()
    .groupby(["Día", "Código", "Nombre"])
    .size()
    .reset_index()
    .rename(columns={0: "TrenesSuprimidos"})
    .sort_values(by=["Día", "TrenesSuprimidos"], ascending=[True, False])
    .copy()
)

trenes_suprimidos_dia_origen = (
    suprimidos.loc[
        (suprimidos["Código"].isin(estaciones)) & (suprimidos["Secuencia"] == 1),
        ["Día", "NTécnico"],
    ]
    .drop_duplicates()
    .groupby(["Día"])
    .size()
    .reset_index()
    .rename(columns={0: "TrenesSuprimidos"})
    .sort_values(by=["Día"], ascending=[True])
    .copy()
)

In [ ]:
display(trenes_suprimidos_estacion_origen.head())
display(trenes_suprimidos_dia_origen.head())

#### En Trayecto

### Mensajería

Incluir solamente los puntos de regulación horaria

controlPointTable a Ramón

#### Calidad Mensajería
- Origen:   alta → salida
- Paso:     aprox → entrada → salida
- Destino:  aprox → entrada

In [ ]:
# Eliminar duplicados
# Lo separo por si en alguno de los movimientos nos queremos qudar con el último duplicado en vez de el primero
sub_dup = [
    "FechaOrigen",
    "NTécnico",
    "Secuencia",
    # "Provincia",
    "Nombre",
    "Código",
    "Movimiento",
]
keep_first = ["PREVISIÓN", "APROXIMACIÓN", "LLEGADA", "SALIDA", "ORIGEN", "FIN", "BAJA"]
keep_last = []
keep = set()
keep = keep | set(
    detalle_movimientos[(detalle_movimientos["Movimiento"].isin(keep_first))]
    .drop_duplicates(subset=sub_dup, keep="first")
    .index
)
keep = keep | set(
    detalle_movimientos[(detalle_movimientos["Movimiento"].isin(keep_last))]
    .drop_duplicates(subset=sub_dup, keep="last")
    .index
)
# detalle_movimientos: pd.DataFrame = detalle_movimientos.drop(discard).reset_index(
#     drop=True
# )

In [ ]:
# Agrupar movimientos del mismo tren
info_trenes = (
    detalle_movimientos.loc[list(keep)]
    .groupby(
        by=[
            "Producto",
            "FechaOrigen",
            "NTécnico",
            "Secuencia",
            # "Provincia",
            "CTC",
            "Nombre",
            "Código",
            # "FuenteMovimiento",
            # "VíaPlanificada",
        ]
    )
    .agg(
        {
            "Fecha": list,
            "Movimiento": list,
            "Vía": list,
        }
    )
    .reset_index()
    .reset_index(drop=True)
)

In [ ]:
info_trenes[info_trenes["NTécnico"] == "00551"].head(1)

In [ ]:
secuencias_correctas = {
    "^(PREVISIÓN_|APROXIMACIÓN_)+LLEGADA_SALIDA$": "PASO_COMPLETO",
    "^LLEGADA_SALIDA$": "PASO",
    "^(PREVISIÓN_|APROXIMACIÓN_)+LLEGADA_(BAJA|FIN$)+": "DESTINO_COMPLETO",
    "^LLEGADA(_BAJA|_FIN$)+": "DESTINO",
    "^ORIGEN_SALIDA$": "ORIGEN",
}
# info_trenes["TipoMovimiento"] = info_trenes["Movimiento"].apply(
#     lambda x: secuencias_correctas.get("_".join(x), "ERROR")
# )
info_trenes["_join_mov"] = info_trenes["Movimiento"].str.join("_")
info_trenes["TipoMovimiento"] = "ERROR"
for k, v in secuencias_correctas.items():
    locs = info_trenes[
        info_trenes["_join_mov"].apply(lambda x: bool(regex.search(k, x)))
    ].index
    info_trenes.loc[locs, "TipoMovimiento"] = info_trenes.loc[
        locs, "_join_mov"
    ].replace(k, v, regex=True)
info_trenes = info_trenes.drop(columns=["_join_mov"])

In [ ]:
def get_num_via(vias):
    num_vias = []
    for el in vias:
        if isEmpty(el):
            continue
        # val = regex.search(r"\d+", el)
        # if val is None:
        #     continue
        # num_vias.append(val.group())
        num_vias.append(el)
    return num_vias


# Solamente tengo en cuenta si hay o no aproximación, no que esta sea correcta
info_trenes["VíaMov"] = (
    info_trenes["Vía"]
    .apply(lambda x: tuple(set(get_num_via(x))))
    .apply(lambda x: x[0] if len(x) == 1 else ("SinVía" if len(x) == 0 else x))
)

# Comprobamos si el movimiento es o no correcto
info_trenes["CORRECTO"] = False
info_trenes.loc[
    np.invert(info_trenes["TipoMovimiento"] == "ERROR")
    & (info_trenes["VíaMov"].apply(lambda x: isinstance(x, str)))
    & np.invert(info_trenes["VíaMov"] == "SinVía"),
    "CORRECTO",
] = True

# Asignamos "vía real" la última vía registrada para los que tienen más de dos vías y decimos que están mal
info_trenes.loc[
    np.invert(info_trenes["VíaMov"].apply(lambda x: isinstance(x, str))), "VíaMov"
] = info_trenes.loc[
    np.invert(info_trenes["VíaMov"].apply(lambda x: isinstance(x, str))), "VíaMov"
].apply(
    lambda x: x[-1]
)

In [ ]:
# Top de provincias por número de movimientos
top_provincias = (
    info_trenes.groupby(
        [
            # "Provincia",
            "CTC",
        ]
    )
    .agg({"NTécnico": "size"})
    .sort_values(by="NTécnico", ascending=False)
    .reset_index()
)

In [ ]:
calidad_mensajeria = (
    info_trenes[
        [
            "FechaOrigen",
            # "Provincia",
            "CTC",
            "Nombre",
            "Código",
            "Movimiento",
            "Vía",
            "TipoMovimiento",
            "VíaMov",
            "CORRECTO",
        ]
    ]
    .reset_index()
    .groupby(
        [
            "FechaOrigen",
            # "Provincia",
            "CTC",
            "Nombre",
            "Código",
            "VíaMov",
        ]
    )
    .agg(
        {
            "index": list,
            "Movimiento": list,
            "Vía": list,
            "TipoMovimiento": Counter,
            "CORRECTO": Counter,
        }
    )
    .reset_index()
)

In [ ]:
df_show = (
    info_trenes.loc[
        # info_trenes["FechaOrigen"] == info_trenes["FechaOrigen"].unique()[0],
        # info_trenes["Provincia"].isin(top_provincias["Provincia"].values[:5]),
        info_trenes["CTC"].isin(top_provincias["CTC"].values[:5]),
        [
            "FechaOrigen",
            # "Provincia",
            "CTC",
            "Nombre",
            "Código",
            "TipoMovimiento",
            "VíaMov",
            "CORRECTO",
        ],
    ]
    .rename(columns={"CORRECTO": "Total"})
    .groupby(
        [
            "FechaOrigen",
            # "Provincia",
            "CTC",
            "Nombre",
            "Código",
            "VíaMov",
            "TipoMovimiento",
        ]
    )
    .agg({"Total": len})
    .reset_index()
    .copy()
)
# df_show["CORRECTO"] = df_show["Total"]
# df_show.loc[df_show["TipoMovimiento"] == "ERROR", "CORRECTO"] = 0
df_show["CORRECTO"] = df_show["Total"] * df_show["TipoMovimiento"].apply(
    lambda x: 0 if x == "ERROR" else 1 if "COMPLETO" in x else 0.75
)
# display(df_show.head())

In [ ]:
levels = [
    "TipoMovimiento",
    "VíaMov",
    "Nombre",
    # "Provincia",
    "CTC",
]
color_columns = ["CORRECTO", "Total"]
value_column = "Total"

# top_node = f"{trafico} - {subdireccion}"
top_node = "España"
df_all_trees = build_hierarchical_dataframe(
    df_show.drop(["FechaOrigen", "Código"], axis=1),
    levels,
    value_column,
    color_columns,
    top_node=top_node,
)
df_all_trees["color"] = df_all_trees["color"].fillna(0)
df_all_trees["color"] = df_all_trees["color"] * 100

In [ ]:
# detalle_movimientos[
#     (detalle_movimientos["Código"] == "17000")
#     & (detalle_movimientos["NTécnico"] == "27311")
# ]

In [ ]:
# info_trenes[
#     (info_trenes["Código"] == "17000")
#     & (info_trenes["VíaMov"] == "2")
#     & (info_trenes["CORRECTO"] == False)
# ]

In [ ]:
traces = []
traces.append(
    go.Treemap(
        ids=df_all_trees["id"],
        labels=df_all_trees["label"],
        parents=df_all_trees["parent"],
        values=df_all_trees["value"],
        branchvalues="total",
        marker=dict(
            colors=df_all_trees["color"],
            colorscale="rdylgn",
            cmid=50,
            cmax=100,
            cmin=40,
        ),
        hovertemplate="<b>%{label} </b> <br> Movimientos: %{value}<br> Correctos: %{color:.2f}%",
        name="",
        maxdepth=3,
        # pathbar_textfont_size=50,
        # textfont_size=20,
    )
)
fig = go.Figure(traces)
fig.update_layout(
    margin=dict(t=20, l=25, r=25, b=25),
)
fig.show()

In [ ]:
df_confusion = (
    detalle_tren[
        [
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Cod. Est",
            "Estación",
            "Vía Teórica",
            "Vía Real",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)
df_confusion[["Vía Teórica", "Vía Real"]] = (
    df_confusion[["Vía Teórica", "Vía Real"]]
    .astype(str)
    .map(lambda x: x.split(".")[0].replace("nan", "SinVía"))
)
df_confusion = df_confusion.groupby(
    by=df_confusion.columns.tolist(), as_index=False
).size()
df_confusion = df_confusion[
    np.invert((df_confusion[["Vía Real", "Vía Teórica"]] == "SinVía").any(axis=1))
]
df_confusion["CORRECTO"] = df_confusion["size"] * (
    df_confusion["Vía Real"] == df_confusion["Vía Teórica"]
).astype(int)
df_confusion = (
    df_confusion.groupby(["Subdirección", "Provincia", "Estación", "Vía Real"])
    .agg({"size": "sum", "CORRECTO": "sum"})
    .reset_index()
    .rename(columns={"size": "Total"})
)
df_confusion.head()

In [ ]:
levels = ["Vía Real", "Estación", "Provincia", "Subdirección"]
color_columns = ["CORRECTO", "Total"]
value_column = "Total"

# top_node = f"{trafico} - {subdireccion}"
top_node = "España"
df_all_trees = build_hierarchical_dataframe(
    df_confusion,
    levels,
    value_column,
    color_columns,
    top_node=top_node,
)
df_all_trees["color"] = df_all_trees["color"].fillna(0)
df_all_trees["color"] = df_all_trees["color"] * 100

traces = []
traces.append(
    go.Treemap(
        ids=df_all_trees["id"],
        labels=df_all_trees["label"],
        parents=df_all_trees["parent"],
        values=df_all_trees["value"],
        branchvalues="total",
        marker=dict(
            colors=df_all_trees["color"],
            colorscale="rdylgn",
            cmid=50,
            cmax=100,
            cmin=40,
        ),
        hovertemplate="<b>%{label} </b> <br> Movimientos: %{value}<br> Correctos: %{color:.2f}%",
        name="",
        maxdepth=4,
        # pathbar_textfont_size=50,
        # textfont_size=20,
    )
)
fig = go.Figure(traces)
fig.update_layout(
    margin=dict(t=20, l=25, r=25, b=25),
)
fig.show()

#### Anticipación Vías en Origen
A partir de `calidad_mensajeria` obtener los movimientos de origen y la diferencia de tiempo salida-alta.

In [ ]:
# (
#     info_trenes[(info_trenes["TipoMovimiento"] == "ORIGEN")]
#     .reset_index(drop=True)
#     .sort_values(by="FechaOrigen")
#     .head()
# )

In [ ]:
anticipacion_origen: pd.DataFrame = (
    info_trenes[(info_trenes["TipoMovimiento"] == "ORIGEN")]
    .reset_index(drop=True)
    .copy()
)
anticipacion_origen["Anticipación (segundos)"] = anticipacion_origen["Fecha"].apply(
    lambda x: (x[1] - x[0]).total_seconds()
)
anticipacion_origen["Anticipación"] = anticipacion_origen[
    "Anticipación (segundos)"
].apply(formatTimedelta)

anticipacion_origen = (
    anticipacion_origen[
        [
            # "Provincia",
            "CTC",
            "Código",
            "Nombre",
            # "Producto",
            "VíaMov",
            "Anticipación (segundos)",
            "Anticipación",
        ]
    ]
    .groupby(
        [
            # "Provincia",
            "CTC",
            "Código",
            "Nombre",
            "VíaMov",
            # "Producto",
        ]
    )
    .agg(list)
    .reset_index()
    .sort_values(by=["Nombre", "VíaMov"])
    .reset_index(drop=True)
)
anticipacion_origen = anticipacion_origen[
    anticipacion_origen["VíaMov"].apply(lambda x: isinstance(x, str))
]
anticipacion_origen = anticipacion_origen.sort_values(
    by="VíaMov", key=lambda x: x.apply(getNumbers)
).reset_index(drop=True)
anticipacion_origen["AnticipaciónMedia (segundos)"] = anticipacion_origen[
    "Anticipación (segundos)"
].apply(np.mean)
anticipacion_origen["AnticipaciónMedia"] = anticipacion_origen[
    "AnticipaciónMedia (segundos)"
].apply(formatTimedelta)

In [ ]:
tiempo_origen_show: pd.DataFrame = anticipacion_origen.loc[
    anticipacion_origen[
        # "Provincia"
        "CTC"
    ].isin(
        top_provincias[
            # "Provincia",
            "CTC"
        ].values[:5]
    ),
    [
        # "Provincia",
        "CTC",
        "Código",
        "Nombre",
        "VíaMov",
        "Anticipación (segundos)",
    ],
].copy()


tiempo_origen_show = tiempo_origen_show.explode("Anticipación (segundos)").reset_index(
    drop=True
)

In [ ]:
def genTracesTOrigen(df: pd.DataFrame):
    df = df.copy()
    df["TiempoAnticipación"] = df["Anticipación (segundos)"].apply(formatTimedelta)
    estaciones_origen = (
        df[["Código", "Nombre"]].drop_duplicates().sort_values(by="Nombre").values
    )
    color_map = sample_random_colors([tuple(el) for el in estaciones_origen])

    traces = []
    x_pos = 1

    ymin = 0
    ymax = roundGroup((df["Anticipación (segundos)"] / 60).max(), 10)
    # xticktext_first_row = []
    # xticktext_second_row = []
    # xtickvals_first_row = []
    # xtickvals_second_row = []
    for c, n in estaciones_origen:

        color = color_map[(c, n)]

        orig_est = df[(df["Código"] == c) & (df["Nombre"] == n)].sort_values(
            by="VíaMov"
        )

        orig_est["Color"] = color

        vias = sortStrNumbers(orig_est["VíaMov"].unique())

        # xticktext_first_row.extend([n] * len(vias))
        # xtickvals_first_row.extend([x_pos + (len(vias) - 1) / 2] * len(vias))

        for v in vias:
            # xticktext_second_row.append(v)
            # xtickvals_second_row.append(x_pos)

            aux_df = orig_est[orig_est["VíaMov"] == v]

            traces.append(
                go.Box(
                    x=[
                        ["<br>".join(splitLongString(n))] * len(aux_df),
                        [v] * len(aux_df),
                    ],
                    y=aux_df["Anticipación (segundos)"] / 60,
                    name=f"{c} - {n}",
                    legendgroup=f"{c} - {n}",
                    showlegend=False,
                    boxpoints="all",
                    jitter=0.1,
                    pointpos=0,
                    marker=dict(
                        color=color,
                        line=dict(color="rgba(0,0,0,0.25)", width=1),
                    ),
                    line=dict(color="rgba(0,0,0,0.25)", width=1),
                    fillcolor=color.replace("rgb(", "rgba(").replace(")", ", 0.45)"),
                    hoverinfo="text",
                    hovertext=setHoverInfo(
                        aux_df, ["Nombre", "VíaMov", "TiempoAnticipación"]
                    ),
                )
            )

            x_pos += 1
        traces.append(
            go.Box(
                x=(None,),
                y=(None,),
                name=f"{c} - {n}",
                legendgroup=f"{c} - {n}",
                showlegend=True,
                line_color=color,
            )
        )

        x_pos += 1

    # xticktext = [xticktext_first_row, xticktext_second_row]
    # xtickvals = [xtickvals_first_row, xtickvals_second_row]
    layout = go.Layout(
        yaxis=dict(
            range=(ymin + 0.1, ymax),
            tickmode="array",
            linecolor="black",
            fixedrange=False,
            showgrid=True,
            zeroline=False,
            title="Anticipación (minutos)",
        ),
        xaxis=dict(
            # tickmode="array",
            # tickvals=xtickvals,
            # ticktext=xticktext,
            fixedrange=False,
            showgrid=False,
            zeroline=False,
            title="Estación-Vía",
        ),
    )

    return traces, layout

In [ ]:
# fig = go.Figure()
# fig.add_trace(
#     go.Box(
#         x=[["BB+"] * 3, ["1", "2", "3"]],
#         y=[1, 2, 3],
#         name=f"1",
#         legendgroup=f"1",
#         showlegend=False,
#         line_color="red",
#     )
# )
# fig.add_trace(
#     go.Box(
#         x=[None], y=[0], name=f"1", legendgroup=f"1", showlegend=True, line_color="red"
#     )
# )
# fig.add_trace(
#     go.Box(
#         x=[["BB"] * 3, ["a", "b", "c"]],
#         y=[6, 5, 4],
#         name=f"2",
#         legendgroup=f"2",
#         showlegend=False,
#         line_color="blue",
#     )
# )
# fig.add_trace(
#     go.Box(
#         x=[None], y=[0], name=f"2", legendgroup=f"2", showlegend=True, line_color="blue"
#     )
# )
# fig.update_layout(
#     xaxis=dict(ticklen=10, ticks="inside", tickformat="%U"),
#     # xaxis2=dict(ticklen=10, ticks="inside", tickformat="%U"),
# )
# fig.show()

In [ ]:
# nrows = 3
# ncols = 2
# fig = make_subplots(rows=nrows, cols=ncols)
for i, p in enumerate(tiempo_origen_show["CTC"].unique()):
    aux_df = tiempo_origen_show[tiempo_origen_show["CTC"] == p]
    fig = go.Figure()
    traces, layout = genTracesTOrigen(
        aux_df[
            aux_df["Código"].isin(
                aux_df[["Código"]]
                .groupby("Código")
                .agg("size")
                .reset_index()
                .sort_values(by=0, ascending=False)["Código"][:10]
            )
        ]
    )
    # fig.add_traces(
    #     traces,
    #     rows=i // ncols +1,
    #     cols=i % ncols +1,
    # )
    # print(i // ncols + 1, i % ncols + 1)
    fig = go.Figure(traces)
    fig.update_layout(
        layout,
        legend=dict(
            groupclick="togglegroup",
            yanchor="middle",
            y=0.5,
        ),
        title=f"Anticipación en origen {p} ",
        modebar=dict(add="v1hovermode"),
        hoverdistance=20,
        paper_bgcolor="#F6F6F6",
        plot_bgcolor="#F6F6F6",
        hoverlabel=dict(bgcolor="white", font_size=16, font_family="consolas"),
    )
    fig.show()
    # break

In [ ]:
# def genHeatmapTOrigen(df: pd.DataFrame, group=1):
#     estaciones_origen = (
#         df[["Código", "Nombre"]].drop_duplicates().sort_values(by="Nombre").values
#     )
#     color_map = sample_random_colors([tuple(el) for el in estaciones_origen])

#     df = df.copy()
#     df["Anticipación (minutos)"] = (df["Anticipación (segundos)"] / 60).astype(int)
#     df["grupo"] = roundGroup(df["Anticipación (minutos)"].values, group=group)

#     fig = go.Figure()
#     z = []
#     x_pos = 0
#     xtickvals = []
#     xticktext = []
#     yticktext = np.array(
#         [
#             f"{el}"
#             for el in np.arange(
#                 0,
#                 df["grupo"].max() + 2 * group,
#                 group,
#             )
#         ]
#     )
#     ytickvals = [i for i, _ in enumerate(yticktext)]
#     hovertext = []
#     x_margin = 3
#     for c, n in estaciones_origen:
#         color = color_map[(c, n)]
#         orig_est = df[(df["Código"] == c) & (df["Nombre"] == n)].sort_values(
#             by="VíaMov"
#         )
#         orig_est["Color"] = color

#         vias = (
#             orig_est["VíaMov"].sort_values(key=lambda x: x.apply(sortElements)).unique()
#         )

#         xtickvals.append(x_pos + (len(vias) * x_margin - 1) / 2)
#         xticktext.append(n)
#         for i, v in enumerate(vias):
#             aux_df = (
#                 orig_est[orig_est["VíaMov"] == v]["grupo"]
#                 .astype(str)
#                 .value_counts()
#                 .reset_index()
#             )

#             z.append(np.full_like(yticktext, np.nan, dtype=float))
#             zeros = np.full_like(yticktext, np.nan, dtype=float)
#             text_data = ["" for _ in range(len(zeros))]
#             vals = aux_df.values
#             for _, i_y, i_g in zip(
#                 *np.intersect1d(yticktext, vals[:, 0], return_indices=True)
#             ):
#                 zeros[i_y] = vals[i_g, 1]
#                 text_data[i_y] = (
#                     f"<b>{c} - {n}</b><br>Vía: {v}<br>Tiempo: {vals[i_g, 1]}"
#                 )
#             z.append(zeros)
#             z.append(np.full_like(yticktext, np.nan, dtype=float))
#             hovertext.append(text_data)
#             x_pos += 3

#         for _ in range(x_margin):
#             z.append(np.full_like(yticktext, np.nan, dtype=float))
#             x_pos += 1
#         fig.add_vline(x=x_pos - x_margin + 1, line_width=1)

#         zeros = np.full_like(yticktext, np.nan, dtype=float)
#         text_data = np.full_like(yticktext, "", dtype=str)
#         hovertext.append(text_data)

#     traces = go.Heatmap(
#         z=np.array(z).T,
#         # hoverinfo="text",
#         hovertext=np.array(hovertext).T,
#         hoverongaps=False,
#         colorscale="rdylgn",
#     )

#     tickmode = "array"
#     layout = go.Layout(
#         yaxis=dict(
#             tickmode=tickmode,
#             tickvals=ytickvals,
#             ticktext=yticktext,
#             linecolor="black",
#             fixedrange=False,
#             showgrid=False,
#             zeroline=False,
#         ),
#         xaxis=dict(
#             tickmode=tickmode,
#             tickvals=xtickvals,
#             ticktext=xticktext,
#             fixedrange=False,
#             showgrid=False,
#             zeroline=False,
#         ),
#     )
#     return traces, layout

In [ ]:
# # nrows = 3
# # ncols = 2
# # fig = make_subplots(rows=nrows, cols=ncols)
# for i, p in enumerate(tiempo_origen_show["Provincia"].unique()):
#     fig = go.Figure()
#     traces, layout = genHeatmapTOrigen(
#         tiempo_origen_show[tiempo_origen_show["Provincia"] == p], 10
#     )
#     # fig.add_traces(
#     #     traces,
#     #     rows=i // ncols +1,
#     #     cols=i % ncols +1,
#     # )
#     # print(i // ncols + 1, i % ncols + 1)
#     fig.add_traces(traces)

#     fig.update_layout(
#         layout,
#         # yaxis_type="log"
#         # xaxis=layout["xaxis"],
#         # yaxis_type="log",
#         # hovermode="x unified",
#     )
#     fig.show()
#     break

#### Tiempos Inferencias
Tiempos medios de Inferencia de vía de CTC (por vía, por relación según acceso a estación...)

In [ ]:
tiempo_inferencia = (
    info_trenes[
        (
            info_trenes["TipoMovimiento"].isin(
                [
                    "PASO_COMPLETO",
                    "DESTINO_COMPLETO",
                    "PASO",
                    "DESTINO",
                ]
            )
        )
        & (info_trenes["CORRECTO"])
    ]
    .reset_index(drop=True)
    .copy()
)
tiempo_inferencia["Anticipación (segundos)"] = 0
tiempo_inferencia.loc[
    tiempo_inferencia["TipoMovimiento"].apply(lambda x: "_COMPLETO" in x),
    "Anticipación (segundos)",
] = tiempo_inferencia.loc[
    tiempo_inferencia["TipoMovimiento"].apply(lambda x: "_COMPLETO" in x), "Fecha"
].apply(
    lambda x: (x[1] - x[0]).total_seconds()
)
tiempo_inferencia["Anticipación"] = tiempo_inferencia["Anticipación (segundos)"].apply(
    formatTimedelta
)

tiempo_inferencia = (
    tiempo_inferencia[
        [
            "Provincia",
            "Código",
            "Nombre",
            # "Producto",
            "VíaMov",
            "Anticipación (segundos)",
            "Anticipación",
        ]
    ]
    .groupby(
        [
            "Provincia",
            "Código",
            "Nombre",
            "VíaMov",
            # "Producto",
        ]
    )
    .agg(list)
    .reset_index()
)
tiempo_inferencia = tiempo_inferencia.sort_values(
    by="VíaMov", key=lambda x: x.apply(getNumbers)
).reset_index(drop=True)
tiempo_inferencia["AnticipaciónMedia (segundos)"] = tiempo_inferencia[
    "Anticipación (segundos)"
].apply(np.mean)
tiempo_inferencia["AnticipaciónMedia"] = tiempo_inferencia[
    "AnticipaciónMedia (segundos)"
].apply(formatTimedelta)

In [ ]:
tiempo_inferencia_show: pd.DataFrame = tiempo_inferencia.loc[
    tiempo_inferencia["Provincia"].isin(top_provincias["Provincia"].values[:5]),
    ["Provincia", "Código", "Nombre", "VíaMov", "Anticipación (segundos)"],
].copy()


tiempo_inferencia_show = tiempo_inferencia_show.explode(
    "Anticipación (segundos)"
).reset_index(drop=True)

In [ ]:
for i, p in enumerate(tiempo_inferencia_show["Provincia"].unique()):
    aux_df = tiempo_inferencia_show[tiempo_inferencia_show["Provincia"] == p]
    fig = go.Figure()
    traces, layout = genTracesTOrigen(
        aux_df[
            aux_df["Código"].isin(
                aux_df[["Código"]]
                .groupby("Código")
                .agg("size")
                .reset_index()
                .sort_values(by=0, ascending=False)["Código"][:10]
            )
        ]
    )
    fig.add_traces(traces)

    fig.update_layout(
        layout,
        legend=dict(
            groupclick="togglegroup",
            yanchor="middle",
            y=0.5,
        ),
        title=f"Anticipación en origen {p} - {trafico}",
        modebar=dict(add="v1hovermode"),
        hoverdistance=20,
        paper_bgcolor="#F6F6F6",
        plot_bgcolor="#F6F6F6",
        hoverlabel=dict(bgcolor="white", font_size=16, font_family="consolas"),
    )
    fig.show()
    # break

In [ ]:
# estaciones_colaterales = {
#     "17000": [
#         "18002",  # Nuevos Ministerios
#         "97201",  # Ramón y Cajal
#         "97101",  # Fuencarral - Fuente Grande
#         "98003",  # Fuente de la Mora
#     ]
# }
# movimientos_colaterales = (
#     df_logs.loc[
#         (df_logs["Código"].isin(estaciones_colaterales["17000"] + ["17000"])),
#         ["Fecha", "Movimiento", "Producto", "FechaOrigen", "NTécnico"]
#         + ["SalidaPlanificada", "LlegadaPlanificada", "Secuencia", "Nombre"]
#         + ["Código", "FuenteVía", "Vía", "FuenteMovimiento"],
#     ]
#     .reset_index(drop=True)
#     .sort_values(by=["NTécnico", "Producto", "Fecha"])
#     .copy()
# )


# tiempos_split = np.split(
#     movimientos_colaterales,
#     np.where(
#         (
#             ~movimientos_colaterales["NTécnico"].eq(
#                 movimientos_colaterales["NTécnico"].shift()
#             )
#         )
#     )[0][1:],
# )
# tiempo_inferencia = []
# for sp in tiempos_split:
#     cods = sp["Código"].unique()
#     if len(cods) >= 2 and cods[1] == "17000":
#         tiempo_inferencia.append(sp)

## MIE

In [ ]:
source = "mie_mse"
dir_logs = Path(f"C:/Users/jose.espinosa/Documents/Data/{source}/PRO")

start_date = "2025-01-26"
end_date = "2025-01-27"
# estaciones = sorted(list(set(["17000", "18000"])))

log_processor = LogProcessor()
w_logs = getFilesByDate(dir_logs, start_date, end_date)
if not w_logs:
    print("No hay logs en los días seleccionados")
else:
    fnames, full_days = list(zip(*w_logs))
    days = f"{full_days[0].strftime('%Y-%m-%d')} - {full_days[-1].strftime('%Y-%m-%d')}"
    # Procesamos
    df_logs = log_processor.loadFilesLogs(
        fnames,
        source,
        load=["tren"],
        mtype=["ocupa", "libera", "alta", "baja"],
        days=days,
        estaciones=estaciones,
        format_fechas=False,
    )
    # df_logs = df_logs[
    #     (df_logs["Fecha"] >= pd.to_datetime(start_date))
    #     & (df_logs["Fecha"] <= pd.to_datetime(end_date))
    # ]
# # Tomo que hace la aproximación, pero esta no tiene por qué ser correcta, así que descarto su vía
# df_logs.loc[df_logs["Movimiento"] == "APROXIMACIÓN", "Vía"] = np.nan
# df_logs.loc[df_logs["Vía"].apply(isEmpty), "Vía"] = None

In [ ]:
# df_logs.head()

In [ ]:
# df_logs[(df_logs["Código"] == "18000") & (df_logs["Vía"].notna())]